In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
def extract_dex_locations(suffix: str):
    pokedex_df = spark.sql(f"""
        SELECT 
            *
        FROM 
            {SILVER_DATABASE_PREFIX}.pokedex__{suffix}                                        
    """)

    pokemon_locations_df = spark.sql(f"""
        SELECT 
            *
        FROM 
            {SILVER_DATABASE_PREFIX}.pokemon_locations__{suffix}                           
    """)

    return pokedex_df, pokemon_locations_df

In [0]:
def get_optimal_encounter_route(pokemon_locations_df, rare_threshold=10.0, synergy_multiplier=2.5):
    """
    1. Clean & Weight Raw Encounters
    ├─ Cap rates at 100%
    ├─ Map Encounter Method Tiers based on the speed at which pokemon are encountered
    └─ Apply Dynamic Condition Penalties - any additional conditions are penalised to prevent encounter being locked behind events
            │
    2. Identify Bottleneck Hubs
    ├─ Mark Rare/Bottleneck Pokémon (<= 2 locations OR rate <= 10%)
    └─ Isolate single best hub location per rare target
            │
    3. Calculate Co-location Synergy
    └─ Apply 2.5x synergy multiplier to common Pokémon sharing hub locations
            │
    4. Deterministic Ranking & Selection
    └─ Window by pokemon_name, select rank #1 (breaks ties by score, rate, name)
    """
    # 1. Clean Data & Extract Condition Names
    cond_names = col("encounter_condition_values.name")

    df_clean = pokemon_locations_df.filter(col("encounter_location_area").isNotNull()) \
        .withColumn("encounter_rate_num", col("encounter_rate").cast("float")) \
        .withColumn("rate_capped", when(col("encounter_rate_num") > 100, 100.0).otherwise(col("encounter_rate_num")))

    # 2. Calculate Base Encounter Score
    df_weighted = df_clean.withColumn(
        "method_weight",
        when(col("encounter_method").isin("static", "gift", "gift-egg", "npc-trade"), 2.0)
        .when(col("encounter_method").isin("walk", "overworld", "dark-grass", "yellow-flowers", "red-flowers", "purple-flowers", "seaweed"), 1.2)
        .when(col("encounter_method").isin("surf", "overworld-water", "super-rod", "good-rod", "old-rod"), 1.0)
        .when(col("encounter_method").isin("horde", "grass-spots", "cave-spots", "bridge-spots", "bubbling-spots", "surf-spots", "super-rod-spots"), 0.9)
        .when(col("encounter_method").isin("rough-terrain", "headbutt", "honey-tree", "rock-smash"), 0.7)
        .otherwise(0.5)
    ).withColumn(
        "condition_penalty",
        when(col("encounter_condition_values").isNull() | (size(col("encounter_condition_values")) == 0), 1.0)
        .when(array_contains(cond_names, "radar") | array_contains(cond_names, "swarm") | array_contains(cond_names, "repel"), 0.4)
        .otherwise(0.7)
    ).withColumn(
        "base_score", col("rate_capped") * col("method_weight") * col("condition_penalty")
    )

    # 3. PASS 1: Identify Priority Hubs from Rare Bottleneck Species
    rare_rank_window = Window.partitionBy("pokemon_pokedex_number").orderBy(
        col("base_score").desc(),
        col("rate_capped").desc(),
        col("encounter_location_area").asc()
    )

    # Lock only the single best area for rare species (Tynamo selects B2F over 1F)
    mandatory_visited_areas = df_weighted.filter(col("rate_capped") <= 15.0) \
        .withColumn("rare_rank", row_number().over(rare_rank_window)) \
        .filter(col("rare_rank") == 1) \
        .select("encounter_location_area") \
        .distinct() \
        .withColumn("is_visited_hub", lit(True))

    # 4. PASS 2: Join Visited Hub Status & Apply Hub Boost
    df_scored = df_weighted.join(mandatory_visited_areas, on="encounter_location_area", how="left") \
        .fillna({"is_visited_hub": False})

    df_final_scored = df_scored.withColumn(
        "synergy_multiplier",
        when(col("is_visited_hub"), lit(100.0)).otherwise(lit(1.0))
    ).withColumn(
        "final_synergy_score", col("base_score") * col("synergy_multiplier")
    )

    # 5. Final Ranking Window per Pokémon
    rank_window = Window.partitionBy("pokemon_pokedex_number").orderBy(
        col("final_synergy_score").desc(),
        col("rate_capped").desc(),
        col("encounter_location_area").asc()
    )

    optimal_locations_df = df_final_scored.withColumn("row_number", row_number().over(rank_window)) \
        .filter(col("row_number") == 1) \
        .select(
            "national_pokedex_number",
            "pokemon_pokedex_number",
            "pokemon_name",
            "encounter_location_area",
            "encounter_condition_values",
            "encounter_method",
            "encounter_rate",
            "encounter_min_level",
            "encounter_max_level",
            col("final_synergy_score").alias("optimal_score"),
            lit(True).alias("is_catchable_wild")
        )

    return optimal_locations_df

In [0]:
silver_pokedex_tables = spark.sql("SHOW TABLES IN silver LIKE 'pokedex*'").select('tableName').distinct()

for pokedex_table_name in [row['tableName'] for row in silver_pokedex_tables.collect()]:
    region_version_suffix = pokedex_table_name.replace('pokedex__', "")
    print(f"Starting optimisations on {region_version_suffix} encounter locations...")

    try:
        pokedex_df, pokemon_locations_df = extract_dex_locations(region_version_suffix)
    except Exception as e:
        print(f"Pokedex or Locations don't exist for {region_version_suffix}. Skipping!")
        continue

    pokemon_locations_df_optimal = get_optimal_encounter_route(pokemon_locations_df)

    optimal_pokedex_locations_df = pokedex_df.alias("px").join(
        pokemon_locations_df_optimal.alias("loc"), 
        on="pokemon_pokedex_number", 
        how="left"
    )

    optimal_pokedex_locations_final_df = (
        optimal_pokedex_locations_df
        .select(
            col("px.national_pokedex_number"),
            col("px.pokemon_name"),
            col("px.generation"),
            col("px.pokemon_pokedex_number"),
            col("loc.encounter_location_area"),
            col("loc.encounter_condition_values"),
            col("loc.encounter_method"),
            col("loc.encounter_rate"),
            col("loc.encounter_min_level"),
            col("loc.encounter_max_level"),
            col("loc.optimal_score")
        )
        .withColumn('encounterable_in_wild', when(col('loc.encounter_location_area').isNull(), False).otherwise(True))
        .orderBy('national_pokedex_number')
    )

    optimal_pokedex_locations_final_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{GOLD_DATABASE_PREFIX}.optimal_pokedex_locations__{region_version_suffix}")

    print(f"Finished optimisations on {region_version_suffix} encounter locations!")